In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')


def estimate_firing_rates(df: pd.DataFrame) -> pd.DataFrame:
    """
    Estimate per-robot firing rates from alliance match data using Poisson regression.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe with columns:
            r1, r2, r3      - team numbers (int or str)
            t1, t2, t3      - shoot time in seconds for each robot (float)
            score           - alliance score for that match (int)

    Returns
    -------
    pd.DataFrame
        One row per robot with columns:
            team            - team number
            firing_rate     - estimated shots per second
            ci_low          - 95% confidence interval lower bound
            ci_high         - 95% confidence interval upper bound
            ci_width        - ci_high - ci_low (uncertainty summary)
            n_matches       - number of matches this robot appeared in
            model           - 'Poisson' or 'NegativeBinomial'

    Raises
    ------
    ValueError
        If required columns are missing or fewer than 2 robots are present.

    Example
    -------
    >>> df = pd.DataFrame({
    ...     'r1': [111, 111, 222], 'r2': [222, 333, 333], 'r3': [333, 444, 444],
    ...     't1': [60, 55, 50],   't2': [50, 60, 65],   't3': [45, 50, 55],
    ...     'score': [210, 195, 220],
    ... })
    >>> estimate_firing_rates(df)
    """
    # ── Validate input ────────────────────────────────────────────────────────
    required = {'r1', 'r2', 'r3', 't1', 't2', 't3', 'score'}
    missing  = required - set(df.columns)
    if missing:
        raise ValueError(f"Input dataframe missing columns: {missing}")

    df = df.copy()

    # Normalise team numbers to clean strings
    for col in ['r1', 'r2', 'r3']:
        df[col] = df[col].apply(lambda x: str(int(float(x))))

    # ── Build design matrix ───────────────────────────────────────────────────
    teams = sorted(set(df['r1']) | set(df['r2']) | set(df['r3']))

    if len(teams) < 2:
        raise ValueError("Need at least 2 distinct robots to estimate rates.")

    X = pd.DataFrame(0.0, index=df.index, columns=teams)
    for idx, row in df.iterrows():
        X.loc[idx, row['r1']] = row['t1']
        X.loc[idx, row['r2']] = row['t2']
        X.loc[idx, row['r3']] = row['t3']

    y          = df['score'].astype(float)
    total_time = X.sum(axis=1)
    offset     = np.log(total_time)
    X_norm     = X.div(total_time, axis=0).fillna(0)
    n_matches  = (X > 0).sum()

    # ── Fit Poisson, fall back to Negative Binomial if overdispersed ──────────
    def _fit_poisson(X_norm, y, offset):
        model  = sm.GLM(y, X_norm, family=sm.families.Poisson(), offset=offset)
        result = model.fit(disp=False)
        od     = result.pearson_chi2 / result.df_resid
        return result, od

    def _fit_nb(X_norm, y, offset, poisson_result):
        mu         = poisson_result.fittedvalues
        alpha_init = max((np.mean(((y - mu) / np.sqrt(mu))**2) - 1) / np.mean(mu), 0.01)
        model      = sm.GLM(y, X_norm,
                            family=sm.families.NegativeBinomial(alpha=alpha_init),
                            offset=offset)
        return model.fit(disp=False), alpha_init

    poisson_result, od = _fit_poisson(X_norm, y, offset)

    if od > 2.0:
        result, _ = _fit_nb(X_norm, y, offset, poisson_result)
        model_used = 'NegativeBinomial'
    else:
        result     = poisson_result
        model_used = 'Poisson'

    # ── Assemble output dataframe ─────────────────────────────────────────────
    ci         = result.conf_int()
    firing_rate = np.exp(result.params)
    ci_low      = np.exp(ci[0])
    ci_high     = np.exp(ci[1])

    out = pd.DataFrame({
        'team':         teams,
        'firing_rate':  firing_rate.values,
        'ci_low':       ci_low.values,
        'ci_high':      ci_high.values,
        'ci_width':     (ci_high - ci_low).values,
        'n_matches':    n_matches.values,
        'model':        model_used,
    })

    return out.sort_values('firing_rate', ascending=False).reset_index(drop=True)


# ── Example usage ─────────────────────────────────────────────────────────────
if __name__ == '__main__':
    import numpy as np

    np.random.seed(42)
    TRUE_RATES = {111: 1.8, 222: 1.2, 333: 2.1, 444: 0.9, 555: 1.5, 666: 1.7}
    robots = list(TRUE_RATES.keys())

    rows = []
    for _ in range(30):
        al = np.random.choice(robots, size=3, replace=False)
        ti = np.random.uniform(40, 80, size=3)
        lam = sum(TRUE_RATES[r] * t for r, t in zip(al, ti))
        rows.append({
            'r1': al[0], 'r2': al[1], 'r3': al[2],
            't1': ti[0], 't2': ti[1], 't3': ti[2],
            'score': np.random.poisson(lam),
        })

    match_df = pd.DataFrame(rows)

    print("Input dataframe (first 5 rows):")
    print(match_df.head())

    result_df = estimate_firing_rates(match_df)

    print("\nEstimated firing rates:")
    print(result_df.to_string(index=False, float_format='{:.3f}'.format))

Input dataframe (first 5 rows):
    r1   r2   r3         t1         t2         t3  score
0  111  222  666  63.946339  46.240746  46.239781    254
1  111  333  444  40.031151  79.688462  64.699260    303
2  222  555  444  45.579754  51.685786  54.654474    180
3  222  555  111  74.397616  67.212302  58.019970    298
4  111  444  333  44.881529  59.807076  41.375541    245

Estimated firing rates:
team  firing_rate  ci_low  ci_high  ci_width  n_matches   model
 333        2.126   1.893    2.387     0.494         12 Poisson
 111        1.969   1.743    2.224     0.482         14 Poisson
 666        1.585   1.415    1.775     0.360         13 Poisson
 555        1.483   1.314    1.674     0.359         18 Poisson
 222        1.221   1.091    1.368     0.277         19 Poisson
 444        0.989   0.882    1.110     0.228         14 Poisson
